In [4]:
#By Eufrasio
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

# 1. CARREGAMENTO DE DADOS
FILE_PATH = "/shared/mauro/Building_Age_Classification/data/preprocessed/preprocessed_data.pkl"
data = pd.read_pickle(FILE_PATH)

X_madrid = data['X_madrid']
y_raw_madrid = data['y_madrid']
pixel_ids = data['pixel_ids_madrid']

# Mapeamento das classes (ex: 1,2,3,4 -> 0,1,2,3)
le = LabelEncoder()
y_madrid = le.fit_transform(y_raw_madrid)

spatial_groups = pd.Series(pixel_ids[:, 0].astype(str) + "_" + pixel_ids[:, 1].astype(str))
num_classes = len(np.unique(y_madrid))

# 2. SEPARAÇÃO ESTRITA: TREINO/VAL (80%) vs TESTE INTOCADO (20%) POR GRUPO ESPACIAL
unique_groups = spatial_groups.unique()
np.random.seed(42)
test_groups = np.random.choice(unique_groups, size=int(len(unique_groups) * 0.2), replace=False)

train_val_mask = ~spatial_groups.isin(test_groups)
test_mask = spatial_groups.isin(test_groups)

X_train_val, y_train_val, groups_train_val = X_madrid[train_val_mask], y_madrid[train_val_mask], spatial_groups[train_val_mask].values
X_test, y_test = X_madrid[test_mask], y_madrid[test_mask]

print("="*60)
print(" DIVISÃO DOS DADOS DE MADRID")
print("="*60)
print(f"Conjunto de Treino/Validação: {X_train_val.shape[0]} amostras")
print(f"Conjunto de Teste Intocado:   {X_test.shape[0]} amostras")

# 3. HIPERPARÂMETROS DO LIGHTGBM
LGBM_PARAMS = {
    'objective': 'multiclass',
    'num_class': num_classes,
    'boosting_type': 'gbdt',
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1
}

# 4. VALIDAÇÃO CRUZADA ESPACIAL (GroupKFold) NO CONJUNTO DE TREINO/VAL
print("\n" + "="*60)
print(" EXECUTANDO VALIDAÇÃO CRUZADA ESPACIAL (5 FOLDS)")
print("="*60)

gkf = GroupKFold(n_splits=5)
val_scores = []
models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_val, y_train_val, groups_train_val)):
    X_tr, y_tr = X_train_val[train_idx], y_train_val[train_idx]
    X_va, y_va = X_train_val[val_idx], y_train_val[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_va, label=y_va, reference=train_data)
    
    model = lgb.train(
        LGBM_PARAMS,
        train_data,
        num_boost_round=500,
        valid_sets=[train_data, val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    preds_val = np.argmax(model.predict(X_va, num_iteration=model.best_iteration), axis=1)
    fold_f1 = f1_score(y_va, preds_val, average='macro')
    val_scores.append(fold_f1)
    models.append(model)
    
    print(f"Fold {fold + 1} - Macro F1 : {fold_f1:.4f}")

print(f"\n Média Macro F1 na Validação: {np.mean(val_scores):.4f} (± {np.std(val_scores):.4f})")

# 5. AVALIAÇÃO FINAL NO CONJUNTO DE TESTE INTOCADO
print("\n" + "="*60)
print(" AVALIAÇÃO FINAL NO CONJUNTO DE TESTE INTOCADO (20%)")
print("="*60)

# Média das predições dos 5 modelos
test_preds_prob = np.zeros((len(X_test), num_classes))
for model in models:
    test_preds_prob += model.predict(X_test, num_iteration=model.best_iteration) / len(models)

final_test_preds = np.argmax(test_preds_prob, axis=1)

print(classification_report(y_test, final_test_preds, target_names=[f"Classe {c}" for c in le.classes_]))

 DIVISÃO DOS DADOS DE MADRID
Conjunto de Treino/Validação: 61011 amostras
Conjunto de Teste Intocado:   15252 amostras

 EXECUTANDO VALIDAÇÃO CRUZADA ESPACIAL (5 FOLDS)
Fold 1 - Macro F1 : 0.6278
Fold 2 - Macro F1 : 0.6298
Fold 3 - Macro F1 : 0.6332
Fold 4 - Macro F1 : 0.6281
Fold 5 - Macro F1 : 0.6295

 Média Macro F1 na Validação: 0.6297 (± 0.0019)

 AVALIAÇÃO FINAL NO CONJUNTO DE TESTE INTOCADO (20%)
              precision    recall  f1-score   support

    Classe 1       0.68      0.43      0.53      2820
    Classe 2       0.60      0.80      0.69      5505
    Classe 3       0.67      0.57      0.62      3663
    Classe 4       0.74      0.67      0.70      3264

    accuracy                           0.65     15252
   macro avg       0.67      0.62      0.63     15252
weighted avg       0.66      0.65      0.64     15252



In [5]:
import joblib

# Guardar os 5 modelos de Madrid e o LabelEncoder no disco
joblib.dump(models, '/shared/mauro/Building_Age_Classification/data/preprocessed/madrid_models.pkl')
joblib.dump(le, '/shared/mauro/Building_Age_Classification/data/preprocessed/label_encoder.pkl')

print(" Modelos de Madrid guardados com sucesso...")

✅ Modelos de Madrid guardados com sucesso no disco!
